In [4]:
# build_mm_qa_1500.py
"""
Build a MULTIMODAL (image + text) evaluation QA set from Code A artifacts.

Inputs:
  - Enriched chunks:
      kaggle/working/enriched_chunks/core/*.json
      kaggle/working/enriched_chunks/longtail/*.json
  - Multimodal pairs (from Code A _save_multimodal_pairs):
      kaggle/working/fine_tuning_data/multimodal_pairs/*_pairs.json

Outputs (evaluation only):
  - kaggle/working/evaluation/qa_sets/mm_eval_<N>.json
  - kaggle/working/evaluation/qa_sets/mm_eval_<N>_instruction.jsonl
  - kaggle/working/evaluation/reports/mm_eval_<N>_manifest.json
"""

import json, re, random, hashlib, argparse, sys
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# ---- Source artifacts from Code A ----
CORE_DIR        = Path("kaggle/working/enriched_chunks/core")
LONGTAIL_DIR    = Path("kaggle/working/enriched_chunks/longtail")
MM_PAIRS_DIR    = Path("kaggle/working/fine_tuning_data/multimodal_pairs")

# ---- Evaluation outputs (kept separate from training) ----
EVAL_BASE_DIR     = Path("kaggle/working/evaluation")
EVAL_QA_DIR       = EVAL_BASE_DIR / "qa_sets"
EVAL_REPORTS_DIR  = EVAL_BASE_DIR / "reports"

# Defaults (overridable)
DEFAULT_TARGET_N = 1500
DEFAULT_OUT_PREFIX = "mm_eval"
DEFAULT_SEED = 2025
MAX_ANS_CHARS = 600
# per-pair questions (usually 1; set 2 if you lack pairs)
QA_PER_MM_PAIR = 1

# Balanced distribution
TARGET_MIX = {
    "disease_definition": 0.10, "pathophysiology": 0.15, "clinical_presentation": 0.20,
    "diagnosis": 0.20, "treatment": 0.20, "epidemiology": 0.10, "case_report": 0.05
}

# Templates specialized for multimodal, but answer must be grounded in (caption + context)
MM_TEMPLATES = {
    "diagnosis": [
        "Using the image and text, what diagnostic method is recommended?",
        "Based on the figure and context, how is the condition confirmed?"
    ],
    "treatment": [
        "According to the image and text, what treatment or regimen is described?",
        "What management approach is supported by the figure and context?"
    ],
    "clinical_presentation": [
        "What clinical features are illustrated by the figure and mentioned in the text?",
        "Which key signs or symptoms are emphasized by the image and context?"
    ],
    "pathophysiology": [
        "What mechanism does the figure convey, as supported by the text?",
        "How does the process shown in the image relate to the described pathophysiology?"
    ],
    "epidemiology": [
        "What epidemiological insight is conveyed by the image and text?",
        "Which populations or geographies are highlighted by the figure and context?"
    ],
    "case_report": [
        "What was the presentation and outcome indicated by the image and text?",
        "What is the key learning point illustrated by this case figure and context?"
    ],
    "disease_definition": [
        "What definition or core characteristics does the figure and text convey?",
        "What essential features of the disease are illustrated here?"
    ],
    "general_medical": [
        "What does the figure illustrate, according to the accompanying text?",
        "Summarize the main point that the image and text convey."
    ]
}

MED_INDICATORS = [
    "treatment","diagnosis","patient","clinical","therapy",
    "disease","condition","Leishmania","leishmaniasis"
]

def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s.strip().lower())
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s

def qa_hash(q: str, a: str, img_paths):
    key = normalize_text(q) + "||" + normalize_text(a[:200]) + "||" + "|".join(sorted(img_paths or []))
    import hashlib
    return hashlib.md5(key.encode()).hexdigest()

def load_chunks_map(include_longtail=True):
    """Return dict[chunk_id] -> chunk (with llm_content_type, keywords, relevance, content, summary)."""
    out = {}
    def _ingest_dir(d):
        if not d.exists(): return
        for p in d.glob("*.json"):
            try:
                items = json.loads(p.read_text(encoding="utf-8"))
                for c in items:
                    out[c["chunk_id"]] = c
            except Exception:
                pass
    _ingest_dir(CORE_DIR)
    if include_longtail:
        _ingest_dir(LONGTAIL_DIR)
    return out

def load_mm_pairs():
    """Load image–chunk pairs written by Code A (_save_multimodal_pairs)."""
    pairs = []
    if not MM_PAIRS_DIR.exists():
        return pairs
    for p in MM_PAIRS_DIR.glob("*_pairs.json"):
        try:
            items = json.loads(p.read_text(encoding="utf-8"))
            pairs.extend(items)
        except Exception:
            pass
    # keep only those with existing image file
    pairs = [r for r in pairs if r.get("image_path") and Path(r["image_path"]).exists()]
    return pairs

def segment_sentences(text: str):
    import re
    sents = re.split(r'(?<=[.!?])\s+', (text or "").strip())
    return [s for s in sents if len(s) > 10]

def extract_answer(question: str, caption: str, context: str, keywords):
    """Ground the answer in caption + context to avoid hallucinations."""
    body = ((caption or "") + " " + (context or "")).strip()
    sents = segment_sentences(body)
    q_tokens = set(normalize_text(question).split())
    best = []
    for s in sents:
        st = normalize_text(s)
        stokens = set(st.split())
        overlap = len(q_tokens & stokens) / (len(q_tokens) + 1e-9)
        kw_score = sum(1 for kw in (keywords or []) if kw and kw.lower() in s.lower()) * 0.3
        med_score = sum(0.2 for m in MED_INDICATORS if m.lower() in s.lower())
        score = overlap + kw_score + med_score
        if score > 0:
            best.append((s, score))
    best.sort(key=lambda x: -x[1])
    if not best:
        fallback = " ".join(sents[:2]) if sents else body[:MAX_ANS_CHARS]
        return fallback[:MAX_ANS_CHARS]
    picked = [t[0] for t in best[:3]]
    ans = " ".join(picked)
    return ans[:MAX_ANS_CHARS]

def make_mm_qa_from_pair(pair, chunk, max_q=1):
    """Generate 1–2 multimodal QAs for a single (image, chunk)."""
    ctype = (chunk.get("llm_content_type") or chunk.get("content_type") or "general_medical")
    templates = MM_TEMPLATES.get(ctype, MM_TEMPLATES["general_medical"])[:max_q]
    caption = pair.get("caption") or chunk.get("llm_summary") or ""
    context = pair.get("context") or chunk.get("content") or ""
    keywords = chunk.get("medical_keywords", [])
    img_path = pair.get("image_path")
    out = []
    for i, q in enumerate(templates):
        a = extract_answer(q, caption, context, keywords)
        if len(normalize_text(a)) < 30:
            continue
        # Require at least medical grounding
        if (not any(tok in a for tok in MED_INDICATORS)
            and not any(kw and kw.lower() in a.lower() for kw in keywords)):
            continue
        out.append({
            "question_id": f"{chunk['chunk_id']}_imgqa_{i}",
            "source_file": pair.get("source_file") or chunk.get("source_file"),
            "chunk_id": chunk.get("chunk_id"),
            "images": [img_path],             # list for future multi-image support
            "question": q,
            "answer": a,
            "context": (caption + " " + context)[:1200],
            "content_type": ctype,
            "question_type": "multimodal_template",
            "relevance_score": float(chunk.get("leishmania_relevance", 0.0)),
            "keywords": keywords,
            "created_at": datetime.now().isoformat(),
            "generation_method": "mm_builder_v1",
            "modality": "image+text"
        })
    return out

def write_outputs(final, out_prefix, target, params):
    EVAL_QA_DIR.mkdir(parents=True, exist_ok=True)
    EVAL_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    out_json   = EVAL_QA_DIR / f"{out_prefix}_{target}.json"
    out_jsonl  = EVAL_QA_DIR / f"{out_prefix}_{target}_instruction.jsonl"
    manifest_p = EVAL_REPORTS_DIR / f"{out_prefix}_{target}_manifest.json"

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for qa in final:
            rec = {
                "instruction": qa["question"],
                "input": (qa.get("context") or "")[:500],
                "output": qa["answer"],
                "metadata": {
                    "source": qa.get("source_file"),
                    "content_type": qa.get("content_type"),
                    "relevance_score": qa.get("relevance_score", 0),
                    "question_type": qa.get("question_type", "multimodal_template"),
                    "modality": qa.get("modality", "image+text"),
                    "images": qa.get("images", []),
                    "for": "evaluation"
                }
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    ct = Counter(q["content_type"] for q in final)
    manifest = {
        "created_at": datetime.now().isoformat(),
        "size": len(final),
        "paths": {"qa_json": str(out_json), "qa_jsonl": str(out_jsonl)},
        "params": params,
        "distribution": dict(ct),
        "modality": "image+text"
    }
    with open(manifest_p, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"✅ Built {len(final)} MM-QA → {out_json}")
    print(f"📝 JSONL (eval-friendly): {out_jsonl}")
    print(f"📄 Manifest: {manifest_p}")
    print("Content-type distribution:", dict(ct))

def build_mm(args):
    random.seed(args.seed)

    # 1) load map chunk_id -> chunk metadata
    chunk_map = load_chunks_map(include_longtail=not getattr(args, "no_longtail", False))
    if not chunk_map:
        raise SystemExit("❌ No enriched chunks found. Run Code A first.")

    # 2) load multimodal pairs
    pairs = load_mm_pairs()
    if not pairs:
        raise SystemExit("❌ No multimodal pairs found. Ensure Code A ran _save_multimodal_pairs().")

    # 3) join + filter by relevance and existence of chunk
    joined = []
    for p in pairs:
        cid = p.get("chunk_id")
        ch = chunk_map.get(cid)
        if not ch: 
            continue
        # prefer leishmania relevance >= 0.4 (longtail+) unless you disable
        rel = float(ch.get("leishmania_relevance", 0.0))
        if getattr(args, "min_relevance", 0.4) is not None and rel < args.min_relevance:
            continue
        joined.append((p, ch))

    if not joined:
        raise SystemExit("❌ No usable (image, chunk) pairs after filtering. Relax min_relevance or rebuild pairs.")

    # 4) generate MM QA
    mm_all = []
    for p, ch in joined:
        mm_all += make_mm_qa_from_pair(p, ch, max_q=min(QA_PER_MM_PAIR, getattr(args, "qa_per_pair", QA_PER_MM_PAIR)))

    # 5) dedup near-duplicates (include image path in hash)
    seen = set()
    dedup = []
    for qa in mm_all:
        h = qa_hash(qa["question"], qa["answer"], qa.get("images", []))
        if h in seen: 
            continue
        seen.add(h)
        dedup.append(qa)

    # 6) balance by content type with soft quotas
    by_type = defaultdict(list)
    for qa in dedup:
        by_type[qa["content_type"]].append(qa)

    final, pool, remaining = [], [], args.target
    for t, frac in TARGET_MIX.items():
        need = int(args.target * frac)
        cand = by_type.get(t, [])
        random.shuffle(cand)
        take = cand[:need]
        final.extend(take)
        remaining -= len(take)
        pool.extend(cand[need:])

    # add other types (general_medical, cross_reference, etc.)
    others = []
    for t, lst in by_type.items():
        if t in TARGET_MIX: 
            continue
        others.extend(lst)
    random.shuffle(pool)
    random.shuffle(others)

    for lst in (pool, others):
        if remaining <= 0: break
        take = lst[:remaining]
        final.extend(take)
        remaining -= len(take)

    if remaining > 0:
        extra = [qa for qa in dedup if qa not in final]
        random.shuffle(extra)
        final.extend(extra[:remaining])

    final = final[:args.target]

    params = {
        "target": args.target,
        "qa_per_pair": getattr(args, "qa_per_pair", QA_PER_MM_PAIR),
        "include_longtail": not getattr(args, "no_longtail", False),
        "min_relevance": getattr(args, "min_relevance", 0.4),
        "seed": args.seed,
        "out_prefix": args.out_prefix
    }
    write_outputs(final, args.out_prefix, args.target, params)

def parse_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--target", type=int, default=DEFAULT_TARGET_N)
    ap.add_argument("--out-prefix", type=str, default=DEFAULT_OUT_PREFIX)
    ap.add_argument("--seed", type=int, default=DEFAULT_SEED)
    ap.add_argument("--qa-per-pair", type=int, default=QA_PER_MM_PAIR)
    ap.add_argument("--no-longtail", action="store_true")
    ap.add_argument("--min-relevance", type=float, default=0.4, help="filter chunks by leishmania_relevance")
    if argv is None and "ipykernel" in sys.modules:
        argv = []  # ignore notebook argv
    args, _ = ap.parse_known_args(argv)
    return args

# Notebook helper
def run_build_mm(target=DEFAULT_TARGET_N, out_prefix=DEFAULT_OUT_PREFIX, seed=DEFAULT_SEED,
                 qa_per_pair=QA_PER_MM_PAIR, include_longtail=True, min_relevance=0.4):
    class _A: pass
    a = _A()
    a.target = int(target)
    a.out_prefix = str(out_prefix)
    a.seed = int(seed)
    a.qa_per_pair = int(qa_per_pair)
    a.no_longtail = (not include_longtail)
    a.min_relevance = float(min_relevance)
    build_mm(a)

if __name__ == "__main__":
    args = parse_args()
    build_mm(args)


✅ Built 1500 MM-QA → kaggle/working/evaluation/qa_sets/mm_eval_1500.json
📝 JSONL (eval-friendly): kaggle/working/evaluation/qa_sets/mm_eval_1500_instruction.jsonl
📄 Manifest: kaggle/working/evaluation/reports/mm_eval_1500_manifest.json
Content-type distribution: {'disease_definition': 95, 'pathophysiology': 254, 'clinical_presentation': 134, 'diagnosis': 377, 'treatment': 230, 'epidemiology': 218, 'case_report': 192}
